In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import json
import math
import random
import logging
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_cosine_schedule_with_warmup
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# =============================================================================
# CONFIG  ← only section you ever need to edit
# =============================================================================

@dataclass
class Config:
    # ── Paths ──────────────────────────────────────────────────────────────────
    base_model_path: str  = "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x"
    train_data_path: str  = "/kaggle/input/deep-past-initiative-machine-translation/train.csv"
    test_data_path: str   = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
    output_dir: str       = "/kaggle/working"

    # ── Training ───────────────────────────────────────────────────────────────
    num_epochs: int           = 6      # trains 6 epochs, saves checkpoint after each
    learning_rate: float      = 5e-6  # conservative — avoids gradient explosion
    batch_size: int           = 2      # low to avoid OOM; grad_accum keeps effective batch=24
    grad_accum_steps: int     = 12     # effective batch = 2 x 12 = 24
    max_input_length: int     = 256    # ByT5 char-level: 256 chars fits fine on P100
    max_target_length: int    = 256
    warmup_ratio: float       = 0.1
    label_smoothing: float    = 0.0  # disabled — was interacting with autocast to cause NaN
    max_grad_norm: float      = 1.0
    val_fraction: float       = 0.05
    seed: int                 = 42
    num_workers: int          = 2
    use_amp: bool             = False  # disabled — causes NaN with fp32 model on P100
    gradient_checkpointing: bool = False  # disabled — was causing training instability

    # ── Augmentation ───────────────────────────────────────────────────────────
    token_dropout_prob: float = 0.05
    token_swap_prob: float    = 0.02

    # ── Ensemble inference ─────────────────────────────────────────────────────
    # Which saved epoch checkpoints to ensemble for final submission.
    # By default uses epochs 2,4,6 — a good spread of early/mid/late.
    # Change to [2,3,4,5,6] to use more models (slower but possibly better).
    ensemble_epochs: List[int] = field(default_factory=lambda: [6])  # only best epoch
    inference_num_beams: int  = 8
    inference_length_penalty: float = 1.3
    inference_batch_size: int = 4

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(exist_ok=True, parents=True)
        if not torch.cuda.is_available():
            self.use_amp = False
        torch.manual_seed(self.seed)
        random.seed(self.seed)
        np.random.seed(self.seed)

    def checkpoint_dir(self, epoch: int) -> Path:
        return Path(self.output_dir) / f"checkpoint_epoch_{epoch}"


# =============================================================================
# LOGGING
# =============================================================================

def setup_logging(out: str) -> logging.Logger:
    Path(out).mkdir(exist_ok=True, parents=True)
    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(Path(out) / "run.log"),
        ],
    )
    return logging.getLogger(__name__)


# =============================================================================
# PREPROCESSING / POSTPROCESSING
# =============================================================================

class Preprocessor:
    def __init__(self):
        self.big_gap   = re.compile(r"(\.{3,}|…+|……)")
        self.small_gap = re.compile(r"(xx+|\s+x\s+)")

    def clean(self, text: str) -> str:
        if pd.isna(text): return ""
        text = str(text)
        text = self.big_gap.sub("<big_gap>", text)
        text = self.small_gap.sub("<gap>", text)
        return text.strip()

    def make_input(self, text: str) -> str:
        return "translate Akkadian to English: " + self.clean(text)


def postprocess(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = text.translate(str.maketrans("ḫḪ", "hH"))
    text = text.translate(str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789"))
    text = re.sub(r"(\[x\]|\(x\)|\bx\b)", "<gap>",     text, flags=re.I)
    text = re.sub(r"(\.{3,}|…|\[\.+\])",  "<big_gap>", text)
    text = text.replace("<gap> <gap>", "<big_gap>")
    text = text.replace("<big_gap> <big_gap>", "<big_gap>")
    text = re.sub(r"\((fem|plur|pl|sing|singular|plural|\?|!)\..\s*\w*\)", "", text, flags=re.I)
    text = text.replace("<gap>", "\x00G\x00").replace("<big_gap>", "\x00B\x00")
    text = text.translate(str.maketrans("", "", '!?()"——<>⌈⌋⌊[]+ʾ/;'))
    text = text.replace("\x00G\x00", " <gap> ").replace("\x00B\x00", " <big_gap> ")
    text = re.sub(r"\b(\w+)(?:\s+\1\b)+", r"\1", text)
    text = re.sub(r"\s+([.,:])", r"\1", text)
    text = re.sub(r"([.,])\1+",  r"\1", text)
    text = re.sub(r"\s+", " ", text).strip().strip("-").strip()
    return text


# =============================================================================
# SCORING  (no sacrebleu — pure Python)
# =============================================================================

def _wng(t, n):
    import collections
    return collections.Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1))

def _cng(s, n):
    import collections
    return collections.Counter(s[i:i+n] for i in range(len(s)-n+1))

def sentence_bleu(h: str, r: str, mx: int = 4) -> float:
    h, r = h.split(), r.split()
    if not h: return 0.0
    bp = 1.0 if len(h) >= len(r) else math.exp(1.0 - len(r) / len(h))
    la = 0.0
    for n in range(1, mx+1):
        hg, rg = _wng(h, n), _wng(r, n)
        c = sum(min(v, rg[k]) for k, v in hg.items())
        la += math.log((c+1) / (max(len(h)-n+1, 0)+1)) / mx
    return bp * math.exp(la) * 100.0

def sentence_chrf(h: str, r: str, co: int = 6, wo: int = 2, b: float = 2.0) -> float:
    def fs(hc, rc):
        m = sum(min(hc[k], rc[k]) for k in hc)
        th, tr = sum(hc.values()), sum(rc.values())
        if not th or not tr: return 0.0
        p, r2 = m/th, m/tr
        d = b**2*p + r2
        return (1+b**2)*p*r2/d if d else 0.0
    pts  = [fs(_cng(h,n), _cng(r,n)) for n in range(1, co+1)]
    pts += [fs(_wng(h.split(),n), _wng(r.split(),n)) for n in range(1, wo+1)]
    return sum(pts)/len(pts)*100.0 if pts else 0.0

def corpus_scores(preds: List[str], refs: List[str]) -> Dict[str, float]:
    bl = [sentence_bleu(p, r) for p, r in zip(preds, refs)]
    ch = [sentence_chrf(p, r) for p, r in zip(preds, refs)]
    mb, mc = sum(bl)/max(len(bl),1), sum(ch)/max(len(ch),1)
    return {
        "BLEU":   round(mb, 2),
        "chrF++": round(mc, 2),
        "Score":  round(math.sqrt(mb*mc) if mb > 0 and mc > 0 else 0, 2),
    }


# =============================================================================
# DATASET
# =============================================================================

class AkkadianDataset(Dataset):
    def __init__(self, df: pd.DataFrame, prep: Preprocessor, augment: bool = False):
        self.augment = augment
        self.inputs  = [prep.make_input(t) for t in df["transliteration"].tolist()]
        self.targets = [str(t) if not pd.isna(t) else "" for t in df["translation"].tolist()]

    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx): return {"input": self.inputs[idx], "target": self.targets[idx]}


# =============================================================================
# COLLATE  (no as_target_tokenizer — works on all transformers versions)
# =============================================================================

def make_collate(tokenizer, max_in: int, max_tgt: int,
                 augment: bool = False, cfg: Optional[Config] = None):
    def collate(batch: List[Dict]):
        src = tokenizer(
            [b["input"]  for b in batch],
            max_length=max_in, padding=True, truncation=True, return_tensors="pt",
        )
        tgt = tokenizer(
            [b["target"] for b in batch],
            max_length=max_tgt, padding=True, truncation=True, return_tensors="pt",
        )
        labels = tgt["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100

        if augment and cfg is not None:
            aug_ids = []
            for ids in src["input_ids"].tolist():
                ids = ids.copy()
                for i in range(len(ids)):
                    if random.random() < cfg.token_dropout_prob:
                        ids[i] = 2  # UNK
                for i in range(len(ids) - 1):
                    if random.random() < cfg.token_swap_prob:
                        ids[i], ids[i+1] = ids[i+1], ids[i]
                aug_ids.append(ids)
            src["input_ids"] = torch.tensor(aug_ids, dtype=torch.long)

        return {
            "input_ids":      src["input_ids"],
            "attention_mask": src["attention_mask"],
            "labels":         labels,
        }
    return collate


# =============================================================================
# LABEL SMOOTHING LOSS
# =============================================================================

class LabelSmoothingLoss(nn.Module):
    def __init__(self, smoothing: float = 0.1, ignore_index: int = -100):
        super().__init__()
        self.smoothing    = smoothing
        self.ignore_index = ignore_index

    def forward(self, logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        if self.smoothing == 0:
            return nn.CrossEntropyLoss(ignore_index=self.ignore_index)(
                logits.view(-1, logits.size(-1)), labels.view(-1))
        vocab = logits.size(-1)
        lp    = torch.nn.functional.log_softmax(logits, dim=-1)
        sv    = self.smoothing / (vocab - 1)
        tgt   = torch.full_like(lp, sv)
        tgt.scatter_(-1, labels.unsqueeze(-1).clamp(min=0), 1.0 - self.smoothing)
        mask  = (labels != self.ignore_index).float()
        return (-(tgt * lp).sum(dim=-1) * mask).sum() / mask.sum().clamp(min=1)


# =============================================================================
# TRAINING
# =============================================================================

def run_training(cfg: Config, logger: logging.Logger) -> List[str]:
    """
    Trains num_epochs epochs from base_model_path.
    Saves a checkpoint after every epoch.
    Returns list of saved checkpoint paths.
    """
    logger.info("="*60)
    logger.info("PHASE 1 — TRAINING")
    logger.info(f"  Base model:   {cfg.base_model_path}")
    logger.info(f"  Epochs:       {cfg.num_epochs}")
    logger.info(f"  LR:           {cfg.learning_rate}")
    logger.info(f"  Eff. batch:   {cfg.batch_size * cfg.grad_accum_steps}")
    logger.info(f"  Label smooth: {cfg.label_smoothing}")
    logger.info("="*60)

    # ── Load model ────────────────────────────────────────────────────────────
    local      = Path(cfg.base_model_path)
    # Load in fp32 with low_cpu_mem_usage — avoids the 2x RAM spike during loading.
    # We use autocast() for fp16 forward passes, but keep params in fp32 so
    # GradScaler works correctly. P100 does not support bfloat16.
    model      = AutoModelForSeq2SeqLM.from_pretrained(
        local,
        low_cpu_mem_usage=True,
    ).to(cfg.device)
    tokenizer  = AutoTokenizer.from_pretrained(local)
    params     = sum(p.numel() for p in model.parameters())
    if cfg.gradient_checkpointing:
        model.gradient_checkpointing_enable()
        logger.info("Gradient checkpointing: ENABLED")
    # Set label smoothing in model config so it's applied inside model.forward()
    # This is numerically stable unlike a custom loss on top of autocast
    model.config.label_smoothing_factor = cfg.label_smoothing
    logger.info(f"Model loaded: {params:,} parameters  label_smoothing={cfg.label_smoothing}")

    # ── Load + split data ─────────────────────────────────────────────────────
    df = pd.read_csv(cfg.train_data_path, encoding="utf-8")
    df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)
    val_df   = df.sample(frac=cfg.val_fraction, random_state=cfg.seed)
    train_df = df.drop(val_df.index).reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)
    logger.info(f"Train: {len(train_df)}  Val: {len(val_df)}")

    prep = Preprocessor()
    train_ds = AkkadianDataset(train_df, prep, augment=True)
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=True, drop_last=False,
        collate_fn=make_collate(tokenizer, cfg.max_input_length,
                                cfg.max_target_length, augment=True, cfg=cfg),
    )
    logger.info(f"Train batches: {len(train_loader)}")

    # ── Optimizer ─────────────────────────────────────────────────────────────
    total_steps  = (len(train_loader) // cfg.grad_accum_steps) * cfg.num_epochs
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    optimizer  = torch.optim.AdamW(model.parameters(),
                                   lr=cfg.learning_rate, weight_decay=0.01)
    scheduler  = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    scaler     = GradScaler(enabled=cfg.use_amp)
    loss_fn    = LabelSmoothingLoss(smoothing=cfg.label_smoothing)
    logger.info(f"Optimizer ready — total_steps={total_steps}  warmup={warmup_steps}")

    # ── Score before training ─────────────────────────────────────────────────
    before = _val_score(model, tokenizer, val_df, prep, cfg)
    logger.info(f"Score BEFORE training: {before}")

    # ── Training loop ─────────────────────────────────────────────────────────
    saved_paths = []
    best_score  = before["Score"]

    for epoch in range(1, cfg.num_epochs + 1):
        logger.info(f"\n── Epoch {epoch}/{cfg.num_epochs} ──")

        # Train
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}", leave=True)
        for step, batch in enumerate(pbar):
            iids  = batch["input_ids"].to(cfg.device)
            amask = batch["attention_mask"].to(cfg.device)
            labs  = batch["labels"].to(cfg.device)

            with autocast(enabled=cfg.use_amp):
                out  = model(input_ids=iids, attention_mask=amask, labels=labs)
                # Use model's built-in loss — numerically stable with autocast/fp32
                loss = out.loss / cfg.grad_accum_steps

            scaler.scale(loss).backward()
            if (step + 1) % cfg.grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            total_loss += loss.item() * cfg.grad_accum_steps
            pbar.set_postfix({
                "loss": f"{total_loss/(step+1):.4f}",
                "lr":   f"{scheduler.get_last_lr()[0]:.2e}",
            })

        train_loss = total_loss / len(train_loader)

        # Validate
        scores = _val_score(model, tokenizer, val_df, prep, cfg)
        logger.info(f"  train_loss={train_loss:.4f}  "
                    f"BLEU={scores['BLEU']}  chrF++={scores['chrF++']}  "
                    f"Score={scores['Score']}")

        if scores["Score"] > best_score:
            best_score = scores["Score"]
            logger.info(f"  ★ New best val score: {best_score:.2f}")

        # Save checkpoint
        ckpt_path = cfg.checkpoint_dir(epoch)
        ckpt_path.mkdir(exist_ok=True, parents=True)
        model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        with open(ckpt_path / "meta.json", "w") as f:
            json.dump({"epoch": epoch, "train_loss": train_loss,
                       "scores": scores, "best_score": best_score}, f, indent=2)
        saved_paths.append(str(ckpt_path))
        logger.info(f"  Saved: {ckpt_path}")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    logger.info(f"\nTraining complete. Best val score: {best_score:.2f}")
    logger.info(f"All checkpoints: {saved_paths}")
    return saved_paths


def _val_score(model, tokenizer, val_df: pd.DataFrame,
               prep: Preprocessor, cfg: Config) -> Dict[str, float]:
    model.eval()
    val_inputs = [prep.make_input(t) for t in val_df["transliteration"].tolist()]
    refs       = [str(t) for t in val_df["translation"].tolist()]
    preds      = []
    for i in range(0, len(val_inputs), 8):
        enc = tokenizer(val_inputs[i:i+8], max_length=512,
                        padding=True, truncation=True, return_tensors="pt")
        with torch.inference_mode():
            with autocast(enabled=cfg.use_amp):
                out = model.generate(
                    input_ids=enc["input_ids"].to(cfg.device),
                    attention_mask=enc["attention_mask"].to(cfg.device),
                    num_beams=4, max_new_tokens=256,
                    length_penalty=1.3, use_cache=True,
                )
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return corpus_scores(preds, refs)


# =============================================================================
# ENSEMBLE INFERENCE
# =============================================================================

def run_ensemble_inference(cfg: Config, checkpoint_paths: List[str],
                           logger: logging.Logger) -> pd.DataFrame:
    """
    Loads all checkpoint models + base model, runs ensemble inference on test set.
    For each input, generates from all models and picks best by chrF++ consensus.
    """
    logger.info("="*60)
    logger.info("PHASE 2 — ENSEMBLE INFERENCE")
    logger.info(f"  Models: {len(checkpoint_paths)}")
    for p in checkpoint_paths:
        logger.info(f"    {p}")
    logger.info("="*60)

    device    = cfg.device
    models    = []
    tokenizer = None

    for path in checkpoint_paths:
        local = Path(path)
        m     = AutoModelForSeq2SeqLM.from_pretrained(
            local,
            low_cpu_mem_usage=True,
        ).eval()  # stays on CPU until needed — moved to GPU one at a time during inference
        models.append(m)
        if tokenizer is None:
            tokenizer = AutoTokenizer.from_pretrained(local)
        params = sum(p.numel() for p in m.parameters())
        logger.info(f"Loaded {path} — {params:,} params")

    logger.info(f"All {len(models)} models loaded")

    prep    = Preprocessor()
    test_df = pd.read_csv(cfg.test_data_path, encoding="utf-8")
    logger.info(f"Test samples: {len(test_df)}")

    ids, texts = test_df["id"].tolist(), test_df["transliteration"].tolist()
    results    = []

    for i in tqdm(range(0, len(texts), cfg.inference_batch_size), desc="Translating"):
        batch_ids   = ids[i:i+cfg.inference_batch_size]
        batch_texts = texts[i:i+cfg.inference_batch_size]
        inputs      = [prep.make_input(t) for t in batch_texts]

        enc   = tokenizer(inputs, max_length=512, padding=True,
                          truncation=True, return_tensors="pt")
        iids  = enc["input_ids"].to(device)
        amask = enc["attention_mask"].to(device)

        gen_kw = dict(
            num_beams=cfg.inference_num_beams,
            max_new_tokens=512,
            length_penalty=cfg.inference_length_penalty,
            use_cache=True,
        )

        # Run models ONE AT A TIME — move to GPU, generate, move back to CPU
        # This avoids holding all 4 models on GPU simultaneously (OOM)
        all_dec = []
        for model in models:
            model.to(device)
            with torch.inference_mode():
                with autocast():
                    out = model.generate(input_ids=iids, attention_mask=amask, **gen_kw)
            all_dec.append(tokenizer.batch_decode(out, skip_special_tokens=True))
            model.to("cpu")  # free GPU memory immediately
            torch.cuda.empty_cache()

        if len(all_dec) == 1:
            results.extend(zip(batch_ids, [postprocess(t) for t in all_dec[0]]))
        else:
            # For each input pick candidate with highest avg chrF++ vs all others
            B = len(batch_ids)
            for b in range(B):
                cands = [all_dec[m][b] for m in range(len(models))]
                scores = []
                for ci, c in enumerate(cands):
                    avg = sum(sentence_chrf(c, cands[j])
                              for j in range(len(cands)) if j != ci)
                    scores.append(avg / max(len(cands)-1, 1))
                best = cands[int(np.argmax(scores))]
                results.append((batch_ids[b], postprocess(best)))

        if torch.cuda.is_available() and i % (cfg.inference_batch_size * 10) == 0:
            torch.cuda.empty_cache()

    df = pd.DataFrame(results, columns=["id", "translation"])
    return df


# =============================================================================
# MAIN
# =============================================================================

def main():
    # ── Clear any leftover GPU memory from previous runs ─────────────────────
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        free = torch.cuda.mem_get_info()[0] / 1e9
        total = torch.cuda.mem_get_info()[1] / 1e9
        print(f"GPU memory free: {free:.1f}GB / {total:.1f}GB")
        if free < 5.0:
            print("WARNING: Less than 12GB free. Please restart the Kaggle kernel")
            print("  Kernel menu → Restart & Clear Output → then run again")

    cfg    = Config()
    logger = setup_logging(cfg.output_dir)

    print(f"PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}  "
              f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

    # ── PHASE 1: Train 6 epochs, save checkpoint after each ──────────────────
    all_checkpoint_paths = run_training(cfg, logger)

    # ── Select checkpoints for ensemble ──────────────────────────────────────
    # Use base model + selected epoch checkpoints
    ensemble_paths = [cfg.base_model_path]  # always include original 33.8 model
    for epoch in cfg.ensemble_epochs:
        ckpt = str(cfg.checkpoint_dir(epoch))
        if Path(ckpt).exists():
            ensemble_paths.append(ckpt)
        else:
            logger.warning(f"Checkpoint for epoch {epoch} not found, skipping")

    logger.info(f"\nEnsemble will use {len(ensemble_paths)} models:")
    for p in ensemble_paths:
        logger.info(f"  {p}")

    # ── PHASE 2: Ensemble inference → submission.csv ─────────────────────────
    results_df = run_ensemble_inference(cfg, ensemble_paths, logger)

    out = Path(cfg.output_dir) / "submission.csv"
    results_df.to_csv(out, index=False)
    logger.info(f"Submission saved: {out}")

    # ── Final report ──────────────────────────────────────────────────────────
    empty   = results_df["translation"].astype(str).str.strip().eq("").sum()
    lengths = results_df["translation"].astype(str).str.len()

    print(f"\n{'='*60}")
    print("COMPLETE")
    print(f"{'='*60}")
    print(f"Translations:  {len(results_df)}")
    print(f"Empty:         {empty}")
    print(f"Length mean:   {lengths.mean():.1f}  min={lengths.min()}  max={lengths.max()}")
    print(f"Models used:   {len(ensemble_paths)}")
    for p in ensemble_paths:
        print(f"  {p}")
    print(f"Submission:    {out}")
    print(f"{'='*60}")

    # Show sample translations
    print("\nSample translations:")
    for idx in [0, len(results_df)//2, len(results_df)-1]:
        if 0 <= idx < len(results_df):
            row = results_df.iloc[idx]
            print(f"  ID {row['id']}: {str(row['translation'])[:80]}")


if __name__ == "__main__":
    main()